#  StoxLSTM: Stochastic Variational LSTM for Time-Series Forecasting

## Introduction
Predicting cryptocurrency prices like **Bitcoin (BTC)** is notoriously difficult due to high volatility and stochastic noise. Standard LSTMs often overfit to noise or fail to capture the probabilistic nature of the market. 

In this notebook, we introduce **StoxLSTM**, a hybrid architecture that combines:
1.  **Patch Embedding:** Borrowed from Vision Transformers (ViT) to process time-series as sequences of patches.
2.  **Stochastic LSTM Cell:** An LSTM unit augmented with a **Variational Autoencoder (VAE)** mechanism directly inside the recurrence. This allows the model to learn a latent distribution $z$ representing market uncertainty.
3.  **Directional Loss:** A custom loss function that penalizes the model heavily if the predicted price direction (up/down) contradicts the actual movement.

##  Mathematical Formulation

### 1. The Stochastic Cell Dynamics
Unlike a standard LSTM where the hidden state $h_t$ is deterministic, StoxLSTM introduces a latent variable $z_t$.

**Standard Gates:**
$$
\begin{aligned}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \\
\tilde{C}_t &= \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)
\end{aligned}
$$

**Latent Variable Injection (VAE):**
At each step, we infer a latent distribution based on the current LSTM state and previous latent state:
$$
\mu_t, \log(\sigma^2_t) = \text{MLP}(h_t, z_{t-1})
$$
Using the reparameterization trick:
$$
z_t = \mu_t + \epsilon \cdot \sigma_t, \quad \text{where } \epsilon \sim \mathcal{N}(0, 1)
$$

**Final Output:**
The output hidden state is a projection of the LSTM state and the sampled latent noise:
$$
h_{final} = \text{Linear}([h_t, z_t])
$$

### 2. Objective Function (Loss)
We optimize a composite loss function containing three terms:

$$
\mathcal{L} = \mathcal{L}_{MSE} + \lambda_{dir} \cdot \mathcal{L}_{Directional} + \beta \cdot D_{KL}
$$

Where:
*   **MSE:** Mean Squared Error for regression accuracy.
*   **Directional Penalty:** Penalizes predictions that move in the opposite direction of the market.
    $$ \mathcal{L}_{Directional} = \text{ReLU}(- \text{sign}(\Delta y_{true}) \cdot \text{sign}(\Delta y_{pred})) $$
*   **KL Divergence:** Regularizes the latent space $z$ to approximate a normal distribution.
    $$ D_{KL} = -0.5 \sum (1 + \log(\sigma^2) - \mu^2 - \sigma^2) $$

---

## Import Lib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import yfinance as yf
import optuna
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datetime import timedelta

##  Data Pipeline & Feature Engineering
We download real-time BTC-USD data via `yfinance`. To make the model robust, we calculate technical indicators:
*   **Log Returns:** Stationarity stabilization.
*   **RSI (Relative Strength Index):** Momentum indicator.
*   **ATR Ratio:** Volatility normalization.
*   **Normalization:** We use a **Dynamic Window Normalization** where each batch is normalized relative to the *first price* in its sequence window ($P_0$).

$$ x_{norm} = \frac{x_t}{x_0} - 1 $$

In [ ]:
# 3. Device Setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on device: {DEVICE}")

# CONFIGURATION

CONFIG = {
    'ticker': 'BTC-USD',
    'period': '2y',
    'interval': '1h',

    'input_seq_len': 96,
    'output_seq_len': 24,

    'train_ratio': 0.80,
    'val_ratio': 0.10,

    'patch_size': 24,
    'stride': 6,

    # 'FEATURE_COLS': ['Close', 'LogRet', 'RSI', 'ATR_Ratio', 'Volume'],
    'FEATURE_COLS': ['Close'],

    # The columns we want to predict (you can specify multiple)
    'TARGET_COLS': ['Close']
}

# Automatically calculate dimensions for the model
CONFIG['input_dim'] = len(CONFIG['FEATURE_COLS'])
CONFIG['output_dim'] = len(CONFIG['TARGET_COLS']) # Number of targets

print(f"Config Loaded. Input Features: {CONFIG['input_dim']}, Targets: {CONFIG['output_dim']}")

In [ ]:
def calculate_indicators(df):
    """
    Calculates technical indicators and cleans data safely.
    """
    df = df.copy()

    # 0. Safety: Replace 0s with NaN to prevent division by zero or infinite logs
    # (Sometimes yfinance returns 0 for missing volume/price)
    cols_to_check = [c for c in ['Close', 'High', 'Low', 'Volume'] if c in df.columns]
    df[cols_to_check] = df[cols_to_check].replace(0, np.nan)
    
    # 1. Log Returns
    # We use numpy validation to suppress warnings for the very first row (which is expected to be NaN)
    with np.errstate(invalid='ignore', divide='ignore'):
        df['LogRet'] = np.log(df['Close'] / df['Close'].shift(1))

    # 2. RSI (14)
    delta = df['Close'].diff()
    # Handle NaNs in delta to prevent RuntimeWarnings in comparisons
    delta = delta.fillna(0) 
    
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-6)
    df['RSI'] = 100 - (100 / (1 + rs))
    df['RSI'] = df['RSI'] / 100.0

    # 3. ATR Ratio
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    atr = true_range.rolling(window=14).mean()
    df['ATR_Ratio'] = atr / df['Close']

    # 4. Volume Clean
    if 'Volume' in df.columns:
        df['Volume'] = df['Volume'].fillna(1.0)

    # Drop NaNs created by indicators (e.g., first 14 rows for RSI/ATR)
    df.dropna(inplace=True)

    # Validate that we actually have the columns we want
    missing_cols = [c for c in CONFIG['FEATURE_COLS'] if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in dataframe: {missing_cols}. Check FEATURE_COLS config.")

    return df[CONFIG['FEATURE_COLS']]

def get_data(ticker, period, interval):
    """
    Downloads stock data with updated yfinance arguments.
    """
    print(f"Downloading data for {ticker}...")
    
    # FIX: Added auto_adjust=True (fixes future warning) 
    # and multi_level_index=False (simplifies column handling)
    try:
        df = yf.download(
            ticker, 
            period=period, 
            interval=interval, 
            progress=False, 
            auto_adjust=True, 
            multi_level_index=False
        )
    except TypeError:
        # Fallback for older yfinance versions
        df = yf.download(ticker, period=period, interval=interval, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df = df.xs(ticker, level=1, axis=1)

    if df.empty:
        raise ValueError("Downloaded DataFrame is empty. Check ticker or internet connection.")

    return calculate_indicators(df)

def create_flexible_batches(data, config):
    input_len = config['input_seq_len']
    output_len = config['output_seq_len']
    features = config['FEATURE_COLS']
    targets = config['TARGET_COLS']
    stride = config.get('stride', 1)

    target_indices = [features.index(t) for t in targets]
    xs, ys, bases = [], [], []
    data_arr = data.values

    # Logic to ensure we don't go out of bounds
    num_samples = (len(data_arr) - input_len - output_len) // stride
    
    for i in range(0, len(data_arr) - input_len - output_len, stride):
        raw_x = data_arr[i : i + input_len].copy()
        raw_y = data_arr[i + input_len : i + input_len + output_len, target_indices].copy()

        # Capture Base (Avoid index errors if Close isn't at 0)
        close_idx = features.index('Close')
        base_price = raw_x[0, close_idx]
        
        # Safety: If base price is 0 (rare but possible in bad data), use 1.0
        if base_price <= 1e-6: base_price = 1.0

        base_vol = 1.0
        if 'Volume' in features:
            vol_idx = features.index('Volume')
            base_vol = raw_x[0, vol_idx]
            if base_vol <= 1e-6: base_vol = 1.0

        # Normalize X
        for f_idx, f_name in enumerate(features):
            if f_name == 'Close':
                raw_x[:, f_idx] = (raw_x[:, f_idx] / base_price) - 1.0
            elif f_name == 'Volume':
                raw_x[:, f_idx] = (raw_x[:, f_idx] / base_vol) - 1.0

        # Normalize Y & Store Bases
        batch_bases = []
        for t_idx, t_name in enumerate(targets):
            if t_name == 'Close':
                raw_y[:, t_idx] = (raw_y[:, t_idx] / base_price) - 1.0
                batch_bases.append(base_price)
            elif t_name == 'Volume':
                raw_y[:, t_idx] = (raw_y[:, t_idx] / base_vol) - 1.0
                batch_bases.append(base_vol)
            else:
                batch_bases.append(1.0)

        xs.append(raw_x)
        ys.append(raw_y)
        bases.append(batch_bases)

    return np.array(xs), np.array(ys), np.array(bases)

# --- Execution ---
try:
    df_rich = get_data(CONFIG['ticker'], CONFIG['period'], CONFIG['interval'])

    n_total = len(df_rich)
    train_split = int(n_total * CONFIG['train_ratio'])
    val_split = int(n_total * (CONFIG['train_ratio'] + CONFIG['val_ratio']))

    train_data = df_rich.iloc[:train_split]
    val_data = df_rich.iloc[train_split:val_split]
    test_data = df_rich.iloc[val_split:]

    X_train, y_train, base_train = create_flexible_batches(train_data, CONFIG)
    X_val, y_val, base_val = create_flexible_batches(val_data, CONFIG)
    X_test, y_test, base_test = create_flexible_batches(test_data, CONFIG)

    print(f"Data Prepared Successfully.")
    print(f"Total Rows Downloaded: {n_total}")
    print(f"Stride: {CONFIG['stride']}")
    print(f"Train Shape: {X_train.shape} (Samples, Input_Seq, Feat_Dim)") 
    print(f"Target Shape: {y_train.shape} (Samples, Out_Seq, Target_Dim)")
except Exception as e:
    print(f"Error during data prep: {e}")

## Visualizing Data Splits

In [ ]:
print("\nVisualizing Data Splits...")


fig = go.Figure()

# 1. Train Trace
fig.add_trace(go.Scatter(
    x=train_data.index,
    y=train_data['Close'],
    name='Train Set',
    line=dict(color='#00FF00', width=1.5)
))

# 2. Validation Trace
fig.add_trace(go.Scatter(
    x=val_data.index,
    y=val_data['Close'],
    name='Validation Set',
    line=dict(color='#FFFF00', width=1.5)
))

# 3. Test Trace
fig.add_trace(go.Scatter(
    x=test_data.index,
    y=test_data['Close'],
    name='Test Set',
    line=dict(color='#FF0000', width=1.5)
))

fig.update_layout(
    title=f'<b>Data Splits: {CONFIG["ticker"]} (Rich Features)</b>',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    template='plotly_dark',
    legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center"),
    hovermode='x unified'
)

fig.show(renderer='iframe')

##  Model Architecture Implementation
Here we define the `StosLSTMCell` and the wrapper `StoxLSTMModel`. Note the `reparameterize` function which enables backpropagation through the stochastic sampling process.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_size, input_dim, embed_dim, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.projection = nn.Linear(patch_size * input_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout) # Added

    def forward(self, x):
        B, L, C = x.shape
        num_patches = L // self.patch_size
        x = x.view(B, num_patches, self.patch_size * C)
        x = self.projection(x)
        x = self.norm(x)
        return self.dropout(x) # Applied

class StosLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size, latent_size, dropout=0.2):
        super(StosLSTMCell, self).__init__()
        self.hidden_size = hidden_size
        self.latent_size = latent_size

        # LSTM gate layers
        self.weight_ih = nn.Linear(input_size, 4 * hidden_size)
        self.weight_hh = nn.Linear(hidden_size, 4 * hidden_size)

        # VAE section
        self.z_mlp = nn.Sequential(
            nn.Linear(hidden_size + latent_size, hidden_size),
            nn.SiLU(),
            nn.Dropout(dropout), # Added
            nn.Linear(hidden_size, 2 * latent_size)
        )
        self.out_proj = nn.Linear(hidden_size + latent_size, hidden_size)
        self.dropout = nn.Dropout(dropout) # Added

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, state):
        h_prev, c_prev, n_prev, z_prev = state

        gates = self.weight_ih(x) + self.weight_hh(h_prev)
        z_raw, i_raw, f_raw, o_raw = gates.chunk(4, 1)

        z_act = torch.tanh(z_raw)
        f = torch.sigmoid(f_raw)
        i = torch.exp(i_raw)
        o = torch.sigmoid(o_raw)

        c_next = f * c_prev + i * z_act
        n_next = f * n_prev + i
        h_next_raw = o * (c_next / (n_next + 1e-6))

        # VAE Part
        z_input = torch.cat([h_next_raw, z_prev], dim=1)
        z_params = self.z_mlp(z_input)
        mu, logvar = z_params.chunk(2, 1)
        z_next = self.reparameterize(mu, logvar)

        # Output Projection
        out_input = torch.cat([h_next_raw, z_next], dim=1)
        h_final = self.out_proj(out_input)
        h_final = self.dropout(h_final) # Applied

        return h_final, (h_next_raw, c_next, n_next, z_next), (mu, logvar)


class StoxLSTMModel(nn.Module):
    def __init__(self, config, embed_dim, hidden_size, latent_size, dropout=0.3):
        super(StoxLSTMModel, self).__init__()
        input_dim = config['input_dim']
        output_dim = config['output_dim'] # Number of targets
        self.output_seq_len = config['output_seq_len']
        self.output_dim = output_dim

        self.patch_embed = PatchEmbedding(config['patch_size'], input_dim, embed_dim, dropout=0.1)
        self.cell = StosLSTMCell(embed_dim, hidden_size, latent_size, dropout=dropout)

        self.hidden_size = hidden_size
        self.latent_size = latent_size

        num_patches = config['input_seq_len'] // config['patch_size']

        # Head: Output size = Seq_Len * Num_Targets
        self.head = nn.Sequential(
            nn.Linear(num_patches * hidden_size, hidden_size // 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, self.output_seq_len * output_dim) # Important change
        )

    def forward(self, x):
        x_emb = self.patch_embed(x)
        batch_size, num_patches, _ = x_emb.size()

        h = torch.zeros(batch_size, self.hidden_size).to(x.device)
        c = torch.zeros(batch_size, self.hidden_size).to(x.device)
        n = torch.ones(batch_size, self.hidden_size).to(x.device)
        z = torch.zeros(batch_size, self.latent_size).to(x.device)

        all_h = []
        kl_loss = 0

        for t in range(num_patches):
            x_t = x_emb[:, t, :]
            h_final, (h, c, n, z), (mu, logvar) = self.cell(x_t, (h, c, n, z))
            all_h.append(h_final)

            kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
            kl_loss += kl.mean()

        all_h = torch.stack(all_h, dim=1)
        all_h_flat = all_h.view(batch_size, -1)

        # Raw Prediction
        prediction_flat = self.head(all_h_flat)

        # Reshape to (Batch, Time, Features)
        prediction = prediction_flat.view(batch_size, self.output_seq_len, self.output_dim)

        return prediction, kl_loss / num_patches

print("Model Architecture defined")


# CUSTOM DIRECTIONAL LOSS FUNCTION


class DirectionalMSELoss(nn.Module):
    def __init__(self, penalty_weight=5.0):
        super(DirectionalMSELoss, self).__init__()
        self.mse = nn.MSELoss()
        self.penalty_weight = penalty_weight

    def forward(self, pred, actual):
        # 1. Standard MSE (for numerical accuracy)
        loss_mse = self.mse(pred, actual)

        # 2. Directional Penalty

        interaction = -1.0 * (pred * actual)
        loss_dir = torch.mean(torch.relu(interaction))

        return loss_mse + (self.penalty_weight * loss_dir)

In [ ]:

# HELPER FUNCTIONS

def train_epoch(model, loader, optimizer, criterion, beta, device):
    model.train()
    epoch_loss = 0
    for b_x, b_y in loader:
        b_x, b_y = b_x.to(device), b_y.to(device)
        optimizer.zero_grad()

        preds, kl_loss = model(b_x)
        mse = criterion(preds, b_y)
        loss = mse + (beta * kl_loss)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)

def evaluate_model(model, loader, base_prices, device):
    """
    Reconstructs Real Prices: Price = Base * (1 + Norm_Val)
    """
    model.eval()
    all_preds = []
    all_actuals = []

    with torch.no_grad():
        for b_x, b_y in loader:
            b_x = b_x.to(device)
            preds, _ = model(b_x)
            all_preds.append(preds.cpu().numpy())
            all_actuals.append(b_y.numpy())

    all_preds = np.concatenate(all_preds)   # Shape (N, 24) -> Normalized
    all_actuals = np.concatenate(all_actuals)

    # --- RECONSTRUCTION LOGIC ---
    # We need to reshape base_prices to broadcast correctly
    # base_prices is (N,) -> convert to (N, 1)
    bases = base_prices.reshape(-1, 1)

    # Formula: Price = Base * (Prediction + 1)
    real_preds = bases * (all_preds + 1.0)
    real_actuals = bases * (all_actuals + 1.0)

    # Flatten for metrics
    flat_preds = real_preds.flatten()
    flat_actuals = real_actuals.flatten()

    # Metrics
    mse = mean_squared_error(flat_actuals, flat_preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(flat_actuals, flat_preds)
    r2 = r2_score(flat_actuals, flat_preds)

    # Directional Accuracy
    diff_act = np.diff(flat_actuals)
    diff_pred = np.diff(flat_preds)
    da = np.mean(np.sign(diff_act) == np.sign(diff_pred)) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "DA": da
    }, real_preds, real_actuals

print("Helpers Defined (Reconstruction Mode).")

## Hyperparameter Optimization with Optuna
We use **Optuna** to find the optimal balance between the reconstruction loss and the KL-Divergence penalty ($\beta$), as well as standard parameters like Learning Rate and Hidden Dimensions.

In [ ]:
# OPTUNA OPTIMIZATION

print("Starting Hyperparameter Optimization...")
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # 1. Suggest Hyperparameters
    params = {
        'embed_dim': trial.suggest_categorical('embed_dim', [32, 64, 128]),
        'hidden_size': trial.suggest_categorical('hidden_size', [64, 128, 256]),
        'latent_size': trial.suggest_int('latent_size', 8, 32),
        'beta': trial.suggest_float('beta', 1e-5, 1e-2, log=True), 
        'lr': trial.suggest_float('lr', 1e-4, 5e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64])
    }

    penalty_w = trial.suggest_float('dir_penalty', 1.0, 10.0)

    # 2. Setup DataLoaders (Only X and y needed for training loop)
    # Convert to Tensors
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))

    train_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=params['batch_size'], shuffle=False)

    # 3. Setup Model
    model = StoxLSTMModel(CONFIG, params['embed_dim'], params['hidden_size'], params['latent_size']).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=params['lr'])
    criterion = DirectionalMSELoss(penalty_weight=penalty_w)

    # 4. Training Loop (Short: 5 Epochs)
    for epoch in range(5):
        # Train
        train_epoch(model, train_loader, optimizer, criterion, params['beta'], DEVICE)

        # Validation (Calculate Loss on Normalized Data)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b_x, b_y in val_loader:
                b_x = b_x.to(DEVICE)
                b_y = b_y.to(DEVICE)
                preds, _ = model(b_x)
                val_loss += criterion(preds, b_y).item()

        avg_val_loss = val_loss / len(val_loader)

        # Pruning
        trial.report(avg_val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return avg_val_loss

# Run Study
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=12) # 12 Trials

best_params = study.best_params
print(f"\nBest Params Found: {best_params}")

## Final Training & Evaluation
We train the model with the best parameters found. The evaluation metric includes **Directional Accuracy (DA)**, which is crucial for trading strategies.

In [ ]:
# 4. Training Loop
epochs = 100
loss_history = []
# ==========================================
# CELL 7: FINAL MODEL TRAINING (CORRECTED)
# ==========================================
print("\nTraining Final Model with Best Parameters...")

# 1. Merge Train + Validation (to have the most data possible)
X_merged = np.concatenate((X_train, X_val))
y_merged = np.concatenate((y_train, y_val))

# 2. Create a secure Internal Split (to prevent test data leakage)
# 90% for final training, 10% for scheduler tuning
split_idx = int(len(X_merged) * 0.90)

X_final_train = X_merged[:split_idx]
y_final_train = y_merged[:split_idx]

X_internal_val = X_merged[split_idx:]
y_internal_val = y_merged[split_idx:]

# 3. Create DataLoaders
# Training loader (shuffle on)
train_ds = TensorDataset(torch.tensor(X_final_train, dtype=torch.float32), torch.tensor(y_final_train, dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=best_params['batch_size'], shuffle=True)

# Internal validation loader (only for the scheduler)
val_ds_internal = TensorDataset(torch.tensor(X_internal_val, dtype=torch.float32), torch.tensor(y_internal_val, dtype=torch.float32))
val_loader_scheduler = DataLoader(val_ds_internal, batch_size=32, shuffle=False)

# 4. Initialize Model & Optimizer
final_model = StoxLSTMModel(
    CONFIG,
    embed_dim=best_params['embed_dim'],
    hidden_size=best_params['hidden_size'],
    latent_size=best_params['latent_size']
).to(DEVICE)

criterion = DirectionalMSELoss(penalty_weight=best_params['dir_penalty'])
optimizer = optim.AdamW(final_model.parameters(), lr=best_params['lr'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 5. Training Loop
for epoch in range(epochs):
    # Train
    train_loss = train_epoch(final_model, train_loader, optimizer, criterion, best_params['beta'], DEVICE)
    loss_history.append(train_loss)

    # Validation (on the Internal Val, not the Test set)
    final_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for b_x, b_y in val_loader_scheduler: # Using the secure loader
            b_x = b_x.to(DEVICE)
            b_y = b_y.to(DEVICE)
            p, _ = final_model(b_x)
            val_loss += criterion(p, b_y).item()

    avg_val_loss = val_loss / len(val_loader_scheduler)
    scheduler.step(avg_val_loss) # This is now safe to do

    if (epoch + 1) % 5 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | LR: {lr:.6f}")

# 5. Plot
plt.figure(figsize=(10, 4))
plt.plot(loss_history, label='Training Loss (Normalized)', color='blue')
plt.title('Final Training Convergence')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("Final Model Trained.")

## TRAIN vs TEST EVALUATION

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---------------------------------------------------------
# 0. SAFETY CHECK: Find the Model Variable
# ---------------------------------------------------------
if 'model' in globals():
    target_model = model
    print("Found variable: 'model'")
elif 'final_model' in globals():
    target_model = final_model
    print("Found variable: final_model")
else:
    raise ValueError("Model not found! Please run the TRAINING CELL (Cell 6/7) first.")

# ---------------------------------------------------------
# 1. EVALUATION FUNCTION (Handles 3D Output)
# ---------------------------------------------------------
def evaluate_model(model_obj, loader, base_prices, device, target_name='Close'):
    model_obj.eval()
    all_preds = []
    all_actuals = []

    # Find target index
    if 'TARGET_COLS' in CONFIG:
        target_idx = CONFIG['TARGET_COLS'].index(target_name)
    else:
        target_idx = 0 # Default if config missing

    with torch.no_grad():
        for b_x, b_y in loader:
            b_x = b_x.to(device)
            preds, _ = model_obj(b_x)
            all_preds.append(preds.cpu().numpy())
            all_actuals.append(b_y.numpy())

    all_preds = np.concatenate(all_preds)   # (N, Seq, Feat)
    all_actuals = np.concatenate(all_actuals)

    # Extract specific target column
    preds_target = all_preds[:, :, target_idx]
    actuals_target = all_actuals[:, :, target_idx]

    # Handle Base Prices Shape
    # base_prices might be (N,) or (N, Feat) depending on previous steps
    if base_prices.ndim > 1:
        bases_target = base_prices[:, target_idx].reshape(-1, 1)
    else:
        # Fallback for old single-dim version
        bases_target = base_prices.reshape(-1, 1)

    # Reconstruction: Base * (Pred + 1)
    real_preds = bases_target * (preds_target + 1.0)
    real_actuals = bases_target * (actuals_target + 1.0)

    # Metrics
    flat_p = real_preds.flatten()
    flat_a = real_actuals.flatten()

    mse = mean_squared_error(flat_a, flat_p)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(flat_a, flat_p)
    r2 = r2_score(flat_a, flat_p)

    # Directional Accuracy
    diff_act = np.diff(flat_a)
    diff_pred = np.diff(flat_p)
    da = np.mean(np.sign(diff_act) == np.sign(diff_pred)) * 100

    return {"RMSE": rmse, "MAE": mae, "R2": r2, "DA": da}, real_preds, real_actuals

# ---------------------------------------------------------
# 2. PREPARE DATA & RUN
# ---------------------------------------------------------
print("Calculating Metrics...")

# Re-create Merged Data
X_final = np.concatenate((X_train, X_val))
y_final = np.concatenate((y_train, y_val))
base_final = np.concatenate((base_train, base_val))

# Loaders
train_eval_ds = TensorDataset(torch.tensor(X_final, dtype=torch.float32), torch.tensor(y_final, dtype=torch.float32))
train_eval_loader = DataLoader(train_eval_ds, batch_size=64, shuffle=False)

test_eval_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))
test_eval_loader = DataLoader(test_eval_ds, batch_size=64, shuffle=False)

# Evaluate
print(">> Evaluating Train Set...")
train_metrics, train_preds, train_actuals = evaluate_model(target_model, train_eval_loader, base_final, DEVICE, 'Close')

print(">> Evaluating Test Set...")
test_metrics, test_preds, test_actuals = evaluate_model(target_model, test_eval_loader, base_test, DEVICE, 'Close')

# ---------------------------------------------------------
# 3. REPORT
# ---------------------------------------------------------
df_report = pd.DataFrame({
    'Metric': ['RMSE ($)', 'MAE ($)', 'R2 Score', 'Directional Acc (%)'],
    'Train Set': [
        f"{train_metrics['RMSE']:.2f}", f"{train_metrics['MAE']:.2f}",
        f"{train_metrics['R2']:.4f}", f"{train_metrics['DA']:.2f}%"
    ],
    'Test Set': [
        f"{test_metrics['RMSE']:.2f}", f"{test_metrics['MAE']:.2f}",
        f"{test_metrics['R2']:.4f}", f"{test_metrics['DA']:.2f}%"
    ],
    'Diff (Test - Train)': [
        f"{test_metrics['RMSE'] - train_metrics['RMSE']:.2f}",
        f"{test_metrics['MAE'] - train_metrics['MAE']:.2f}",
        f"{test_metrics['R2'] - train_metrics['R2']:.4f}",
        f"{test_metrics['DA'] - train_metrics['DA']:.2f}%"
    ]
})

print("\n FINAL MODEL DIAGNOSTICS:")
df_report

## Recursive Forecasting & Visualization
The model predicts the next 24 hours. To forecast longer horizons (e.g., 7-30 days), we use a **Recursive Strategy**:
1. Predict the next step.
2. Feed the prediction back as input for the next step.
3. Repeat.

Below is the complete interactive chart showing history, test set validation, and future forecast.

In [ ]:
# ROBUST RECURSIVE FORECAST

def generate_forecast(final_model, df_hist, config, days=7):
    """
    Generates a multi-step recursive forecast.
    """
    final_model.eval()
    input_len = config['input_seq_len']
    output_len = config['output_seq_len']
    features = config['FEATURE_COLS']
    targets = config['TARGET_COLS']

    current_df = df_hist.copy() # We will append to this
    future_preds = [] # Store only predictions

    iterations = int(np.ceil((24 * days) / output_len))

    for i in range(iterations):
        # 1. Get Input Window
        raw_window = current_df[features].iloc[-input_len:].values.copy()

        # 2. Normalize Input
        # Must match the logic in create_flexible_batches
        base_price = raw_window[0, features.index('Close')]
        base_vol = raw_window[0, features.index('Volume')] if 'Volume' in features else 1.0
        if base_vol == 0: base_vol = 1.0

        norm_window = raw_window.copy()
        for idx, name in enumerate(features):
            if name == 'Close': norm_window[:, idx] = (raw_window[:, idx] / base_price) - 1.0
            elif name == 'Volume': norm_window[:, idx] = (raw_window[:, idx] / base_vol) - 1.0

        # 3. Predict
        inp = torch.tensor(norm_window, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            p, _ = final_model(inp) # Shape: (1, 24, n_targets)
            p = p.cpu().numpy().reshape(output_len, -1)

        # 4. De-normalize & Store
        # Create a DataFrame for the new predictions
        new_rows = []
        for step in range(len(p)):
            row_vals = {}
            # Start with last known values for everything (Naive approach for non-targets)
            last_known = current_df.iloc[-1]
            for col in features:
                row_vals[col] = last_known[col]

            # Overwrite targets with AI predictions
            for t_idx, t_name in enumerate(targets):
                val_norm = p[step, t_idx]
                if t_name == 'Close': val_real = base_price * (val_norm + 1.0)
                elif t_name == 'Volume': val_real = base_vol * (val_norm + 1.0)
                else: val_real = val_norm

                row_vals[t_name] = val_real

                # Keep track for final output (only Close usually matters for plotting)
                if t_name == 'Close':
                    future_preds.append(val_real)

            new_rows.append(row_vals)

            # Trick: Append one by one or batch?
            # Appending one by one allows updating "last_known" if we had autoregressive logic.
            # Here we batch update current_df at the end of the window for speed,
            # BUT for the next step in the loop, 'raw_window' needs these values.

        # Append to history for the next iteration
        new_df = pd.DataFrame(new_rows)
        current_df = pd.concat([current_df, new_df], ignore_index=True)

    return future_preds[:24*days]

In [ ]:
Day = 7
preds = generate_forecast(final_model, df_rich, CONFIG, days=Day)
print(f"Forecast Done. Last Price: {preds[-1]:.2f}")

In [ ]:
# ======================================================
# MASTER VISUALIZATION: COMPLETE HISTORY + FUTURE
# ======================================================
import plotly.graph_objects as go
from datetime import timedelta
import numpy as np

print("Generating Full History Visualization...")

# ------------------------------------------------------
# 1. SETUP & CHECKS
# ------------------------------------------------------
if 'model' in globals(): target_model = model
elif 'final_model' in globals(): target_model = final_model
else: raise ValueError("Model not found! Run training first.")

# Check for Future Forecast
if 'preds' in globals(): 
    future_forecast = preds
elif 'forecast_prices' in globals(): 
    future_forecast = forecast_prices
else: 
    print("Warning: No forecast found. Using dummy extension.")
    future_forecast = [df_rich['Close'].iloc[-1]] * 24

# Configuration
input_len = CONFIG['input_seq_len']
output_len = CONFIG['output_seq_len']
# Ensure we use the stride used in batch creation (defaults to 1 if not set)
stride = CONFIG.get('stride', 1) 
target_col_idx = CONFIG['TARGET_COLS'].index('Close') if 'TARGET_COLS' in CONFIG else 0

# ------------------------------------------------------
# 2. CALCULATE ALIGNED TEST PREDICTIONS
# ------------------------------------------------------
print(">> Aligning Test Predictions with Dates...")
target_model.eval()

ai_pred_prices = []
ai_pred_dates = []

# Global start index of the test set in the original dataframe
test_start_global_idx = val_split

with torch.no_grad():
    # Process in batches
    batch_size = 64
    for i in range(0, len(X_test), batch_size):
        # Slice batch
        current_X = X_test[i : i + batch_size]
        current_base = base_test[i : i + batch_size]
        
        # Tensor conversion
        t_X = torch.tensor(current_X, dtype=torch.float32).to(DEVICE)
        
        # Predict
        raw_pred, _ = target_model(t_X)
        raw_pred = raw_pred.cpu().numpy() # (Batch, Seq, Feat)
        
        # Reconstruct & Align
        for k in range(len(current_X)):
            # 1. Denormalize Price
            # Handle base shape (N,) vs (N, 1)
            b_price = current_base[k, target_col_idx] if current_base.ndim > 1 else current_base[k]
            pred_seq = raw_pred[k, :, target_col_idx]
            
            real_price_seq = b_price * (pred_seq + 1.0)
            ai_pred_prices.extend(real_price_seq)
            
            # 2. Find Exact Dates
            # Logic: (Batch Index * Stride) + Input_Len = Start of Prediction Window
            idx_in_test_set = (i + k) * stride
            
            # Global index in df_rich
            # Note: We used `test_data` to generate X_test
            start_target_idx = idx_in_test_set + input_len
            end_target_idx = start_target_idx + output_len
            
            # Safety check
            if end_target_idx <= len(test_data):
                # Get dates from the test_data dataframe
                seq_dates = test_data.index[start_target_idx : end_target_idx]
                ai_pred_dates.extend(seq_dates)

# Trim to match lengths (handles potential edge cases)
limit = min(len(ai_pred_prices), len(ai_pred_dates))
ai_pred_prices = ai_pred_prices[:limit]
ai_pred_dates = ai_pred_dates[:limit]

# ------------------------------------------------------
# 3. PREPARE FUTURE FORECAST DATA
# ------------------------------------------------------
last_real_date = df_rich.index[-1]
last_real_price = df_rich['Close'].iloc[-1]

# Generate future dates
future_dates = [last_real_date + timedelta(hours=h+1) for h in range(len(future_forecast))]

# Connect the drawing (Start from last real point)
display_future_dates = [last_real_date] + future_dates
display_future_prices = [last_real_price] + list(future_forecast)

# ------------------------------------------------------
# 4. PLOT ALL DATA
# ------------------------------------------------------
fig = go.Figure()

# A. ENTIRE Training Data
# We use [:val_split] to show everything from start to the validation split
fig.add_trace(go.Scatter(
    x=df_rich.index[:val_split],
    y=df_rich['Close'].values[:val_split],
    name='Training Data (History)',
    line=dict(color='#555555', width=1), # Dark gray for history
    opacity=0.7
))

# B. Test Actual
# From split to end
fig.add_trace(go.Scatter(
    x=test_data.index,
    y=test_data['Close'],
    name='Test Actual (Ground Truth)',
    line=dict(color='#00FF00', width=1.5) # Green
))

# C. Test Prediction (AI)
fig.add_trace(go.Scatter(
    x=ai_pred_dates,
    y=ai_pred_prices,
    name='Test Prediction (AI)',
    line=dict(color='#FF3333', width=1), # Red
    opacity=0.9
))

# D. Future Forecast
fig.add_trace(go.Scatter(
    x=display_future_dates,
    y=display_future_prices,
    name='Future Forecast',
    mode='lines+markers',
    line=dict(color='#00FFFF', width=2), # Cyan
    marker=dict(size=3)
))

# ------------------------------------------------------
# 5. LAYOUT & SLIDER
# ------------------------------------------------------
fig.update_layout(
    title=f'<b>{CONFIG["ticker"]} Complete Analysis</b>',
    xaxis=dict(
        title='Date',
        type="date",
        rangeslider=dict(visible=True), # <--- SLIDER ADDED HERE
        rangeselector=dict(
            buttons=[
                dict(count=7, label="1W", step="day", stepmode="backward"),
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(step="all", label="ALL"),
            ]
        )
    ),
    yaxis=dict(title='Price (USD)', fixedrange=False),
    template='plotly_dark',
    legend=dict(orientation="h", y=1.1, x=0.5, xanchor="center"),
    height=800,
    hovermode='x unified'
)

# Vertical line for "NOW"
fig.add_vline(x=last_real_date.timestamp() * 1000, line_width=1, line_dash="dash", line_color="white", annotation_text="Now")

fig.show(renderer='iframe')